<a href="https://colab.research.google.com/github/AKChumba/AKChumba/blob/main/NAT820S_Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NAT820S — Practical Lab: Neural Networks, Autoencoders & Transformers

**Weeks 8 & 10 | Natural Language Processing and Applications**

In this lab you'll get hands-on with everything from Lecture 5:
- A simple neural network (LSTM) text classifier
- A text autoencoder that learns compressed representations
- Fine-tuning a pre-trained transformer (DistilBERT) for classification

Look out for **🔧 TODO** cells — that's where you write code. Everything else is a worked example you can run and adapt.

> Reference: Gupta, Majumder & Vajjala (2020), *Practical Natural Language Processing*, O'Reilly, Chapter 1 & Chapter 4.

**How to use this notebook:** Run each cell in order (Shift+Enter). Read the markdown before each code cell — it tells you what the code does and what you need to do next.

**Note on runtime:** For faster training, go to *Runtime → Change runtime type → T4 GPU* before you start.

## Part 0 — Setup

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

!pip install transformers datasets evaluate accelerate -q
import transformers
print("TensorFlow version:", tf.__version__)
print("Transformers version:", transformers.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

## Part 1 — A Simple Neural Network Text Classifier

We'll build a small LSTM-based classifier on the IMDB movie review dataset (built into Keras) — the same dataset the textbook uses for its CNN/LSTM examples.

In [ ]:
# Load the IMDB dataset, keeping only the 10,000 most common words
NUM_WORDS = 10000
MAXLEN = 200

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=NUM_WORDS)

print("Training examples:", len(x_train))
print("Test examples:", len(x_test))
print("Example review (as word indices):", x_train[0][:20])
print("Label (0=negative, 1=positive):", y_train[0])

In [ ]:
# Pad/truncate every review to the same length so we can batch them
x_train_pad = pad_sequences(x_train, maxlen=MAXLEN)
x_test_pad = pad_sequences(x_test, maxlen=MAXLEN)

print("Padded shape:", x_train_pad.shape)

### Build and train the LSTM model

This mirrors the book's Keras LSTM example from Chapter 4 — an embedding layer, then an LSTM layer, then a dense output.

In [ ]:
lstm_model = Sequential([
    Embedding(NUM_WORDS, 128, input_length=MAXLEN),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
lstm_model.summary()

In [ ]:
history = lstm_model.fit(
    x_train_pad, y_train,
    batch_size=64,
    epochs=2,
    validation_split=0.2
)

test_loss, test_acc = lstm_model.evaluate(x_test_pad, y_test)
print("Test accuracy:", test_acc)

### 🔧 TODO 1.1
Build a **CNN** version of this classifier instead of the LSTM, following the book's Chapter 4 pattern: `Embedding` → `Conv1D(128, 5, activation='relu')` → `MaxPooling1D(5)` → `Conv1D(128, 5, activation='relu')` → `GlobalMaxPooling1D()` → `Dense(1, activation='sigmoid')`. Train it for 2 epochs and compare its accuracy and training time to the LSTM above.

In [ ]:
# TODO: build cnn_model using Sequential(), compile it, train it on x_train_pad/y_train,
# and evaluate it on x_test_pad/y_test

cnn_model = Sequential([
    # your layers here
])


### 🔧 TODO 1.2 — Reflection
Which model trained faster, the LSTM or your CNN? Which had higher test accuracy? Does this match what the lecture said about CNNs vs. LSTMs?

**Your answer here:**


## Part 2 — A Text Autoencoder

Now let's build a small autoencoder, following the same "bowtie" shape from the lecture's Figure 1-18 (input → narrow hidden layer → output, same size as input). Instead of movies, we'll compress **Bag-of-Words vectors** of short text snippets and see what the compressed representation captures.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# A small corpus mixing three topics: weather, cooking, and sports
sentences = [
    "the weather today is sunny and warm",
    "it is raining heavily outside today",
    "the forecast predicts snow this weekend",
    "chop the onions and fry them in oil",
    "bake the cake for thirty minutes at high heat",
    "boil the pasta then add the tomato sauce",
    "the striker scored a goal in the first half",
    "the team won the championship match last night",
    "the coach substituted two players in the second half",
]

vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(sentences).toarray().astype("float32")
vocab_size = bow.shape[1]
print("Bag-of-words shape:", bow.shape, "| Vocabulary size:", vocab_size)

In [ ]:
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

encoding_dim = 3  # compress down to just 3 numbers per sentence

input_layer = Input(shape=(vocab_size,))
hidden_layer = Dense(encoding_dim, activation="relu", name="bottleneck")(input_layer)
output_layer = Dense(vocab_size, activation="sigmoid")(hidden_layer)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

In [ ]:
autoencoder.fit(bow, bow, epochs=300, batch_size=3, verbose=0)
print("Training complete. Final loss:", autoencoder.evaluate(bow, bow, verbose=0))

In [ ]:
# Extract just the encoder part (input -> bottleneck) to get compressed vectors
encoder = Model(inputs=input_layer, outputs=hidden_layer)
compressed = encoder.predict(bow)

for sentence, vec in zip(sentences, compressed):
    print(f"{vec.round(2)}  <-  {sentence}")

### 🔧 TODO 2.1
Look at the compressed 3-number vectors above. Do sentences about the **same topic** (weather / cooking / sports) end up with **similar** compressed vectors? Compute the cosine similarity between a couple of pairs to check your intuition.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# TODO: pick two sentences you think are similar (e.g. both about weather) and two you
# think are different (e.g. one weather, one sports), and print their cosine similarity
# using cosine_similarity(compressed[i:i+1], compressed[j:j+1])


### 🔧 TODO 2.2 — Reflection
This autoencoder used plain Bag-of-Words as input, which ignores word order. Based on what you learned about LSTM autoencoders in the lecture, what would change if you used one instead? What kind of input would it expect?

**Your answer here:**


## Part 3 — Demo: Self-Attention From Scratch

Before we use a pre-trained transformer as a black box, let's build **scaled dot-product attention** ourselves with plain NumPy, following the diagram from the lecture:

`Input → Linear (Wq, Wk, Wv) → Q, K, V → MatMul(Q, Kᵀ) → Scale → Softmax → MatMul with V → Attention Output`

This is the exact mechanism inside every transformer layer (including BERT, which we'll use in Part 4).

In [ ]:
import numpy as np

np.random.seed(42)

# A tiny "sentence" of 4 tokens, each already embedded as a 8-dimensional vector
# (in a real transformer these would come from an embedding layer)
tokens = ["The", "animal", "was", "tired"]
d_model = 8
X = np.random.randn(len(tokens), d_model)

print("Input embeddings shape:", X.shape)
print(X.round(2))

In [ ]:
# Step 1: create the Wq, Wk, Wv weight matrices (normally learned; here just random for the demo)
d_k = d_model  # keeping dimensions equal for simplicity

Wq = np.random.randn(d_model, d_k) * 0.5
Wk = np.random.randn(d_model, d_k) * 0.5
Wv = np.random.randn(d_model, d_k) * 0.5

# Step 2: project the input into Query, Key, and Value matrices
Q = X @ Wq
K = X @ Wk
V = X @ Wv

print("Q shape:", Q.shape, "| K shape:", K.shape, "| V shape:", V.shape)

In [ ]:
# Step 3: compute raw attention scores = Q . K^T
scores = Q @ K.T
print("Raw scores:\n", scores.round(2))

# Step 4: scale by 1/sqrt(d_k) -- this keeps the softmax gradients well-behaved
scaled_scores = scores / np.sqrt(d_k)
print("\nScaled scores:\n", scaled_scores.round(2))

In [ ]:
# Step 5: softmax each row so the attention weights for each token sum to 1
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

attention_weights = softmax(scaled_scores)
print("Attention weights (rows sum to 1):\n", attention_weights.round(2))
print("\nRow sums:", attention_weights.sum(axis=1).round(2))

In [ ]:
# Step 6: weighted sum of Value vectors, using the attention weights
attention_output = attention_weights @ V

print("Attention output shape:", attention_output.shape)
print(attention_output.round(2))

### 🔧 TODO 3.1
Look at the `attention_weights` matrix above. Row `i`, column `j` tells you how much token `i` attends to token `j`. Print out, for the word **"was"** (index 2), which other word it attends to the *most*. Use `np.argmax`.

In [ ]:
# TODO: find which token index "was" (row 2 of attention_weights) attends to most,
# and print the corresponding word from `tokens`


### 🔧 TODO 3.2 — Reflection
This demo used **random** weight matrices, so the attention pattern is meaningless — a real transformer *learns* Wq, Wk, Wv during training so that attention weights reflect actual linguistic relationships (like the "animal"/"it" example from the lecture).

In your own words: why does dividing by `sqrt(d_k)` (Step 4) matter? (Hint: think about what happens to the softmax if the raw dot-product scores get very large as `d_k` grows.)

**Your answer here:**


### Multi-head attention, briefly

Real transformers don't just do this once — they run several attention "heads" in parallel (each with its own Wq, Wk, Wv), then concatenate the results. This lets different heads specialize: one might track subject-verb agreement, another might track coreference (like "it" → "animal"). We won't implement multi-head attention by hand here, but you now understand the core building block every head repeats.



## Part 4 — Fine-Tuning a Pre-Trained Transformer

Training a transformer from scratch needs huge amounts of data and compute — that's exactly why pre-trained models like BERT exist. We'll use **DistilBERT** (a smaller, faster version of BERT) via Hugging Face's `transformers` library, and fine-tune it for sentiment classification — the same idea as the book's `ktrain` BERT example, using a more modern toolkit.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# A small slice of the IMDB dataset (from Hugging Face) to keep training fast for a lab
raw_dataset = load_dataset("imdb")
small_train = raw_dataset["train"].shuffle(seed=42).select(range(500))
small_test = raw_dataset["test"].shuffle(seed=42).select(range(200))

print(small_train[0]["text"][:300])
print("Label:", small_train[0]["label"])

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

train_tokenized = small_train.map(tokenize_function, batched=True)
test_tokenized = small_test.map(tokenize_function, batched=True)

train_tokenized = train_tokenized.remove_columns(["text"]).rename_column("label", "labels")
test_tokenized = test_tokenized.remove_columns(["text"]).rename_column("label", "labels")
train_tokenized.set_format("torch")
test_tokenized.set_format("torch")

print("Datasets ready ✅")

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

accuracy_metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./bert_finetune",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

Notice how few lines of code that was, compared to Part 1 — all the heavy lifting (learning general language understanding from millions of documents) was already done. We're just adapting it to our task.

### 🔧 TODO 4.1
Write **two of your own movie review sentences** — one clearly positive, one clearly negative — and use the fine-tuned model to predict their sentiment. Use the code skeleton below.

In [ ]:
my_reviews = [
    "",  # your positive review
    "",  # your negative review
]

# TODO: tokenize my_reviews with the tokenizer (return_tensors="pt", padding=True, truncation=True, max_length=256),
# then call model(**inputs) to get logits, and use torch.nn.functional.softmax on the logits
# (dim=-1) to see the predicted probabilities for each class
import torch


### 🔧 TODO 4.2 — Reflection
Compare your experience building the LSTM classifier in Part 1 versus fine-tuning DistilBERT in Part 3: lines of code, training time, and (roughly) how confident the predictions felt. Given the lecture's "Is Deep Learning the Silver Bullet?" discussion, when might you still prefer the simpler LSTM (or even a non-DL classifier from Lecture 4) over fine-tuning a transformer?

**Your answer here:**


## Part 5 — Wrap-Up

### 🔧 TODO 5.1 — Summary Table
Build a small `pandas` DataFrame comparing the three approaches from this lab: columns `Approach`, `Training Data Needed`, `Training Time`, `Code Complexity`. Rows: LSTM (from scratch), Autoencoder, Fine-tuned DistilBERT.

In [ ]:
# TODO: build and display the comparison DataFrame
summary_rows = [
    # {"Approach": "LSTM (from scratch)", "Training Data Needed": ..., "Training Time": ..., "Code Complexity": ...},
]

pd.DataFrame(summary_rows)

### Reflection

In 3–4 sentences: imagine you're building a sentiment classifier for a small startup with only 300 labeled customer reviews. Based on everything in this lab and Lecture 5's "silver bullet" discussion, which approach would you start with, and why?

**Your answer here:**

---
*End of lab. Save your notebook (File → Save a copy in Drive) before you leave.*